To solve this lab i have create two different algorithms, one for positive weigths and one for the negatives ones. The positive one is solved with a uniform cost search. I order my border path based on the cost, so that everytime i take an element, it is the most promisisng one. I check if the solution reach the goal at first, before adding the new element to be sure to find the best one.

To solve the negatives ones I have used an uniform cost modify, because now the cost can decrease after we add a new city to the path. So i have eliminated some constraint, creating only a greedy search based on the best one. If the path have negative values, it raise the exception path negative.

In [48]:
from itertools import product, combinations
import numpy as np
!pip  install networkx icecream
import networkx as nx
from icecream import ic

In [49]:
def create_problem(
    size: int,
    *,
    density: float = 1.0,
    negative_values: bool = False,
    noise_level: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """Problem generator for Lab3"""
    rng = np.random.default_rng(seed)
    map = rng.random(size=(size, 2))
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            problem[a, b] += np.sqrt(
                np.square(map[a, 0] - map[b, 0]) + np.square(map[a, 1] - map[b, 1])
            )
        else:
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round()

In [50]:
negative_value = True
problem = create_problem(100, density=0.15, noise_level=40, negative_values= negative_value)

In [51]:
masked = np.ma.masked_array(problem, mask=np.isinf(problem))
G = nx.from_numpy_array(masked, create_using=nx.DiGraph)

In [52]:
solution=([], 0)


def goal_test(state, goal_state):
    return state[-1] == goal_state

def available_actions(problem, state):
    action=[]
    for i in range(len(problem)):
       if problem[state[-1], i] != np.inf and i not in state:
          action.append(i)
    return action

def do_action(problem, state, action):
    new_state = state.copy()
    new_state.append(action)
    return new_state


    

In [53]:
def available_actions_negative(problem, state):
    action=[]
    for i in range(len(problem)):
       if problem[state[-1], i] != np.inf:
          action.append(i)
    return action


In [54]:
def h(state, problem):
    actions = available_actions(problem, state)
    if not actions:
        return float('inf')
    vals = [problem[state[-1], j] for j in actions]
    return float(np.min(vals))

In [55]:
def uniform_cost_search_positive(start_node, goal_node):
    """Perform uniform cost search (Dijkstra-like) to find the lowest-cost path to the goal.

    Returns (path_list, total_cost). If no path found returns (None, np.inf).
    The function expects `problem` as the adjacency cost matrix used elsewhere in the notebook.
    """
    frontier = PriorityQueue()
    # Push initial path (list of nodes) with cost 0
    frontier.push([start_node], 0)
    # best_cost[node] = best found cost to reach node so far
    best_cost = {start_node: 0}

    while not frontier.is_empty():
        path, cost = frontier.pop()
        current = path[-1]

        # If this popped state has cost worse than the best known, skip it
        if cost > best_cost.get(current, float('inf')):
            continue

        if goal_test(path, goal_node):
            return path, cost

        for action in available_actions(problem, path):
            new_path = path + [action]
            new_cost = cost + problem[current, action]

            # If this path improves best known cost for `action`, push to frontier
            if new_cost < best_cost.get(action, float('inf')):
                best_cost[action] = new_cost
                a_star_function = new_cost + h(new_path, problem)
                frontier.push(new_path, a_star_function)

    return None, float('inf')


In [56]:
# Define custom exception for negative cycle detection
class NegativeCycleDetected(Exception):
    """Exception raised when a negative cycle is detected during search."""
    pass
class NegativePathDetected(Exception):
    """Exception raised when a negative path is detected during search."""
    pass

def uniform_cost_search_negative(start_node, goal_node):
    frontier = PriorityQueue()
    # Push initial path (list of nodes) with cost 0
    frontier.push([start_node], 0)
    first=False
    # best_cost[node] = best found cost to reach node so far
    best_cost = {start_node: 0}

    while not frontier.is_empty():
        path, cost = frontier.pop()
        current = path[-1]

        # Detect negative cycle: if path exceeds graph size, there's a cycle
        if len(path) > len(problem[0]) and cost < 0:
            raise NegativeCycleDetected(
                f"Negative cycle detected when searching from {start_node} to {goal_node}. "
                f"Path length {len(path)} exceeds number of nodes {len(problem[0])}."
            )

        if goal_test(path, goal_node) and first==True:
            if cost<0:
                raise NegativePathDetected(
                f"Negative path detected when searching from {start_node} to {goal_node}. "
            )
            return path, cost
        first=True

        for action in available_actions_negative(problem, path):
            new_path = path + [action]
            new_cost = cost + problem[current, action]
            a_star_function = new_cost + h(new_path, problem)
            frontier.push(new_path, a_star_function)
        

    return None, float('inf')



In [57]:
from heapq import heappush, heappop
import itertools
from typing import Callable, Any

class PriorityQueue:
    """Min-priority queue with customizable ordering key.

    Default ordering uses the provided cost. Use `key(item, cost)` to
    compute a numeric priority (e.g. for A* f = g + h).

    Usage:
      from pq import PriorityQueue
      pq = PriorityQueue()
      pq.push(item, cost)
      item, cost = pq.pop()
    """
    def __init__(self, key: Callable[[Any, float], float]=None):
        self.key = key if key is not None else (lambda item, cost: cost)
        self.heap = []
        self._counter = itertools.count()

    def set_key(self, key: Callable[[Any, float], float]):
        self.key = key

    def push(self, item, cost=None):
        # Allow pushing a (item, cost) tuple for backwards compatibility
        if cost is None and isinstance(item, (list, tuple)) and len(item) >= 2:
            obj, c = item[0], item[1]
            item, cost = obj, c
        if cost is None:
            cost = 0
        priority = self.key(item, cost)
        heappush(self.heap, (priority, next(self._counter), item, cost))

    def pop(self):
        if not self.heap:
            raise IndexError("pop from empty PriorityQueue")
        priority, _, item, cost = heappop(self.heap)
        return item, cost

    def peek(self):
        if not self.heap:
            return None
        priority, _, item, cost = self.heap[0]
        return item, cost, priority

    def is_empty(self):
        return not self.heap

    def __len__(self):
        return len(self.heap)

__all__ = ["PriorityQueue"]


In [ ]:

# TEST PARAMETRIZZATO: cicla su più configurazioni e confronta UCS vs NetworkX
sizes = [10, 20, 50]
density_values = [0.2, 0.5, 0.8]
noise_levels = [0, 10, 50]

test_results = []

for size in sizes:
    for dens in density_values:
        for noise in noise_levels:
            for use_negative in [True, False]:
                # Genera il problema con i parametri attuali
                problem = create_problem(size=size, density=dens, noise_level=noise, 
                                        negative_values=use_negative, seed=42)
                
                # Rigenerae il grafo per il nuovo problema
                masked = np.ma.masked_array(problem, mask=np.isinf(problem))
                G = nx.from_numpy_array(masked, create_using=nx.DiGraph)
                
                # Test per start nodes 0, 1, 2
                n = problem.shape[0]
                start_nodes = [0, 1, 2]
                mismatch_count = 0
                config_results = []

                for start_idx in start_nodes:
                    if start_idx >= n:
                        continue
                    
                    for dest_idx in range(n):
                        if start_idx == dest_idx:
                            continue

                        # NetworkX shortest path
                        try:
                            method = 'bellman-ford' if use_negative else 'dijkstra'
                            nx_path = nx.shortest_path(G, start_idx, dest_idx, weight='weight', method=method)
                            nx_cost = nx.path_weight(G, nx_path, weight='weight')
                        except nx.NetworkXNoPath:
                            nx_path = None
                            nx_cost = float('inf')
                        except nx.NetworkXUnbounded:
                            nx_path = None
                            nx_cost = float('-inf')

                        # Our UCS
                        try:
                            if use_negative:
                                ucs_path, ucs_cost = uniform_cost_search_negative(start_idx, dest_idx)
                            else:
                                ucs_path, ucs_cost = uniform_cost_search_positive(start_idx, dest_idx)
                            if ucs_path is None:
                                ucs_cost = float('inf')
                        except NegativeCycleDetected:
                            ucs_path = None
                            ucs_cost = float('-inf')
                        except NegativePathDetected:
                            ucs_path = None
                            ucs_cost = float('-inf')

                        # Comparison
                        match = (nx_cost == ucs_cost) or (math.isinf(nx_cost) and math.isinf(ucs_cost))
                        if not match:
                            mismatch_count += 1
                            config_results.append({
                                "start": start_idx, "dest": dest_idx,
                                "nx_cost": nx_cost, "ucs_cost": ucs_cost
                            })

                test_results.append({
                    "config": {"size": size, "density": dens, "noise": noise, "negative": use_negative},
                    "mismatches": mismatch_count,
                    "details": config_results
                })
                
                print(f"size={size}, dens={dens}, noise={noise}, negative={use_negative}: {mismatch_count} mismatches")

print("\n" + "="*60)
print(f"SUMMARY: Total configs tested = {len(test_results)}")
total_mismatches = sum(r["mismatches"] for r in test_results)
print(f"Total mismatches across all configs = {total_mismatches}")
if total_mismatches == 0:
    print("✓ ALL TESTS PASSED")
else:
    print("✗ Some tests failed. Details in test_results.")
